In [2]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.linear_model import LassoCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import RobustScaler
from scipy.stats import skew
import warnings
warnings.filterwarnings('ignore')

train = pd.read_csv('inputs/train.csv')
test = pd.read_csv('inputs/test.csv')
test_ID = test['Id']
y_train = np.log1p(train['SalePrice'])

all_data = pd.concat([train.drop(['Id', 'SalePrice'], axis=1), test.drop('Id', axis=1)]).reset_index(drop=True)




In [3]:
none_cols = ['PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'MasVnrType']
for col in none_cols: all_data[col] = all_data[col].fillna('None')

zero_cols = ['GarageYrBlt', 'GarageArea', 'GarageCars', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF','TotalBsmtSF', 'BsmtFullBath', 'BsmtHalfBath', 'MasVnrArea']
for col in zero_cols: all_data[col] = all_data[col].fillna(0)

mode_cols = ['MSZoning', 'Electrical', 'KitchenQual', 'Exterior1st', 'Exterior2nd', 'SaleType', 'Functional']
for col in mode_cols: all_data[col] = all_data[col].fillna(all_data[col].mode()[0])

all_data['LotFrontage'] = all_data.groupby('Neighborhood')['LotFrontage'].transform(lambda x: x.fillna(x.median()))
all_data = all_data.drop(['Utilities'], axis=1)

all_data['TotalSF'] = all_data['TotalBsmtSF'] + all_data['1stFlrSF'] + all_data['2ndFlrSF']

all_data['TotalBathrooms'] = all_data['FullBath'] + (0.5 * all_data['HalfBath']) + all_data['BsmtFullBath'] + (0.5 * all_data['BsmtHalfBath'])

all_data['HouseAge'] = all_data['YrSold'] - all_data['YearBuilt']
all_data['RemodAge'] = all_data['YrSold'] - all_data['YearRemodAdd']

all_data['HasPool'] = all_data['PoolArea'].apply(lambda x: 1 if x > 0 else 0)
all_data['Has2ndFloor'] = all_data['2ndFlrSF'].apply(lambda x: 1 if x > 0 else 0)
all_data['HasGarage'] = all_data['GarageArea'].apply(lambda x: 1 if x > 0 else 0)
all_data['HasBsmt'] = all_data['TotalBsmtSF'].apply(lambda x: 1 if x > 0 else 0)
all_data['HasFireplace'] = all_data['Fireplaces'].apply(lambda x: 1 if x > 0 else 0)

all_data['MSSubClass'] = all_data['MSSubClass'].astype(str)
all_data['OverallCond'] = all_data['OverallCond'].astype(str)
all_data['YrSold'] = all_data['YrSold'].astype(str)
all_data['MoSold'] = all_data['MoSold'].astype(str)



In [ ]:
numeric_feats = all_data.dtypes[all_data.dtypes != "object"].index
skewed_feats = all_data.select_dtypes(include=[np.number]).apply(lambda x: skew(x.dropna())).sort_values(ascending=False)

high_skew = skewed_feats[skewed_feats > 0.75]
for feat in high_skew.index:
    all_data[feat] = np.log1p(all_data[feat])

all_data = pd.get_dummies(all_data)
X_train = all_data.iloc[:len(train)]
X_test = all_data.iloc[len(train):]



TypeError: unsupported operand type(s) for /: 'str' and 'int'

In [ ]:
lasso_model = make_pipeline(RobustScaler(), LassoCV(alphas=[0.0001, 0.0003, 0.0006, 0.001, 0.003, 0.006, 0.01, 0.03, 0.06, 0.1, 0.3], random_state=42, cv=5))
lasso_model.fit(X_train, y_train)
lasso_preds = lasso_model.predict(X_test)

xgb_model = XGBRegressor(n_estimators=1000, learning_rate=0.05, max_depth=3, subsample=0.8, colsample_bytree=0.8, random_state=42)
xgb_model.fit(X_train, y_train)
xgb_preds = xgb_model.predict(X_test)

blend_preds = (0.5 * lasso_preds) + (0.5 * xgb_preds)

final_prices = np.expm1(blend_preds)



In [ ]:
submission = pd.DataFrame({
    "Id": test_ID,
    "SalePrice": final_prices
})

submission_filename = 'v2_blended_super_features.csv'
submission.to_csv(submission_filename, index=False)